In [ ]:
import pandas as pd
import numpy as np
import textwrap
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [ ]:
columns = [
    "lp", "v", "GTT", "GTn", "GGn", "Ts", "Tp", "T48", "T1", "T2","P48", "P1", "P2", "Pexh", "TIC", "mf",
    "kMc", "kMt"]

df = pd.read_csv("./UCI CBM Dataset/data.txt", sep=r"\s+", header = None, names = columns)

print(df.head())
#print(df.shape)


In [ ]:
data = np.loadtxt("./UCI CBM Dataset/data.txt")
data

In [ ]:
data[:-2]

Each row corresponds to a snapshot of a ship’s propulsion system under certain operating conditions.
models how engine components degrade over time under different conditions

Dataset structure:
- Samples: ~11,934
- Input features: 16
- Targets (labels): 2 continuous variables

Targets:
- GT Compressor Decay State Coefficient
- GT Turbine Decay State Coefficient
These represent health/degradation levels of the gas turbine components.

In [ ]:
df.hist(figsize = (18, 14), bins = 40)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x="v", y="mf", alpha=0.3)
plt.title("Ship Speed vs Fuel Flow")
plt.show()


NASA Dataset

In [ ]:
cols = (
    ["unit", "time", "os1", "os2", "os3"] +
    [f"s{i}" for i in range(1, 22)]  # sensors s1–s21
)

def load_fd(fd):
    train = pd.read_csv(f"./CMAPSSData/train_FD{fd}.txt", sep=r"\s+", header=None, names=cols)
    test  = pd.read_csv(f"./CMAPSSData/test_FD{fd}.txt",  sep=r"\s+", header=None, names=cols)
    rul   = pd.read_csv(f"./CMAPSSData/RUL_FD{fd}.txt",   sep=r"\s+", header=None, names=["RUL"])
    return train, test, rul

train_fd1, test_fd1, rul_fd1 = load_fd("001")

In [ ]:
def add_train_rul(df):
    max_time = df.groupby("unit")["time"].max()
    df = df.merge(max_time.rename("max_time"), on="unit")
    df["RUL"] = df["max_time"] - df["time"]
    return df.drop(columns=["max_time"])

train_fd1 = add_train_rul(train_fd1)


In [ ]:
def add_test_rul(test, rul):
    last_cycles = test.groupby("unit")["time"].max().reset_index()
    last_cycles["RUL"] = rul["RUL"]
    return test.merge(last_cycles[["unit", "RUL"]], on="unit")

test_fd1 = add_test_rul(test_fd1, rul_fd1)


In [ ]:
train_fd1.head(5)

In [ ]:
test_fd1.head(5)

Sensor Distributions

Each engine (unit) runs through a sequence of cycles:

It starts healthy.
It gradually degrades.
In the training set, it runs all the way to failure.
In the test set, it stops before failure.


So the Remaining Useful Life (RUL) is:


How many cycles remain before the engine fails, starting from the last cycle in the test trajectory.


In [ ]:
train_fd1.iloc[:, 5:].hist(figsize = (18, 14), bins = 40)
plt.tight_layout()
plt.show()


In [ ]:
rul_1_gt = pd.read_csv('./CMAPSSData/RUL_FD001.txt', header = None)
rul_1_gt.rename(columns={0: "RUL_1gt"}, inplace = True)

column_name_dict = { 0: "engine_id", 1: "cycle", 2: "op_set_1", 3: "op_set_2", 4: "op_set_3", 5:"sensor_1", 6: "sensor_2",
                   7: "sensor_3", 8: "sensor_4", 9: "sensor_5", 10: "sensor_6", 11: "sensor_7", 12: "sensor_8", 13: "sensor_9",
                  14: "sensor_10", 15: "sensor_11", 16: "sensor_12", 17: "sensor_13", 18: "sensor_14", 19: "sensor_15", 20: "sensor_16",
                  21: "sensor_17", 22: "sensor_18", 23: "sensor_19", 24: "sensor_20", 25: "sensor_21", 26: "sensor_22", 27: "sensor_23"}

test_1 = pd.read_csv('./CMAPSSData/test_FD001.txt', header = None, sep = ' ')
test_1.rename(columns = column_name_dict, inplace = True)

train_1 = pd.read_csv('./CMAPSSData/train_FD001.txt', header = None, sep = ' ')
train_1.rename(columns = column_name_dict, inplace = True)

In [ ]:
train_1

In [ ]:
train_1.drop(columns = ["sensor_22", "sensor_23"], inplace = True)
test_1.drop(columns = ["sensor_22", "sensor_23"], inplace = True)

In [ ]:
train_1.head(3)

In [ ]:
rul_1 = pd.DataFrame(train_1.groupby('engine_id')['cycle'].max()).reset_index()
rul_1.columns = ['engine_id', 'max']

rul_train_1 = train_1.merge(rul_1, on=['engine_id'], how='left')
rul_train_1['RUL'] = rul_train_1['max'] - rul_train_1['cycle']
rul_train_1.drop(['max'], axis=1, inplace = True)

In [ ]:
rul_1_gt["engine_id"]=rul_1_gt.index + 1

max_1 = pd.DataFrame(test_1.groupby('engine_id')['cycle'].max()).reset_index()
max_1.columns = ['engine_id', 'max']
max_test_1 = test_1.merge(max_1, on=['engine_id'], how='left')
rul_test_1 = max_test_1.merge(rul_1_gt, on=['engine_id'], how='left')

rul_test_1['RUL'] = rul_test_1['max'] - rul_test_1['cycle'] + rul_test_1["RUL_1gt"] 
rul_test_1.drop(['max', 'RUL_1gt'], axis=1, inplace=True)

In [ ]:
rul_train_1.groupby('engine_id')['cycle'].describe()

In [ ]:
rul_train_1.describe().transpose()

In [ ]:
deleted_1 = ["op_set_3", "sensor_18", "sensor_19"]

rul_train_1.drop(columns = deleted_1, inplace = True)
rul_test_1.drop(columns = deleted_1, inplace = True)

In [ ]:
df_corr = rul_train_1.drop(columns=["engine_id"]).corr()

# correlation matrix
sns.set(font_scale = 0.8)
plt.figure(figsize = (24,16))
sns.heatmap(df_corr, annot=True, fmt=".4f",vmin=-1, vmax=1, linewidths=.5, cmap = sns.color_palette("coolwarm", 200))

plt.figtext(.45, 0.9,'correlation matrix of train_1', fontsize=16, ha='center')
plt.xticks(rotation=90)
plt.show()

In [ ]:
rul_train_1.head(3)

In [ ]:
rul_test_1.head(3)

In [ ]:
corr_del=["sensor_1", "sensor_5", "sensor_10", "sensor_16"]
rul_train_1.drop(columns = corr_del, inplace = True)
rul_test_1.drop(columns = corr_del, inplace = True)

In [ ]:
print("Min RUL:", rul_train_1["RUL"].min())
print("Max RUL:", rul_train_1["RUL"].max())
print(rul_train_1["RUL"].describe())


In [ ]:
print(rul_train_1["RUL"].value_counts(normalize = True).sort_index())

In [ ]:
def rul_to_class_fd001(r):
    if r > 155:
        return 2 
    elif r > 51:
        return 1 
    else:
        return 0 

rul_train_1["label"] = rul_train_1["RUL"].apply(rul_to_class_fd001)
rul_test_1["label"] = rul_test_1["RUL"].apply(rul_to_class_fd001)


rul_train_1.head(3)

In [ ]:
rul_train_1.drop(['RUL'], axis = 1, inplace = True)
rul_test_1.drop(['RUL'], axis = 1, inplace = True)

In [ ]:
np.save("./CMAPSSData/train_01.npy", rul_train_1)
np.save("./CMAPSSData/test_01.npy", rul_test_1)

In [ ]:
import numpy as np
import builtins

for path in ["./CMAPSSData/train_01.npy", "./CMAPSSData/test_01.npy"]:
    arr = np.load(path, allow_pickle=True)
    print("\nPATH:", path)
    print("type:", type(arr))
    print("dtype:", getattr(arr, "dtype", None))
    print("shape:", getattr(arr, "shape", None))

    if isinstance(arr, np.ndarray) and arr.dtype == object:
        print("first element type:", type(arr.flat[0]))
        if arr.shape == ():
            obj = arr.item()
            print("scalar object type:", type(obj))
            if isinstance(obj, builtins.dict):
                print("dict keys:", list(obj.keys()))
            elif isinstance(obj, tuple):
                print("tuple length:", len(obj))
            elif isinstance(obj, list):
                print("list length:", len(obj))
      

In [ ]:
plt.plot(rul_train_1["RUL"])

In [ ]:
rul_test_1.columns

CMAPSS FD002 dataset

In [ ]:
rul_2_gt = pd.read_csv('./CMAPSSData/RUL_FD002.txt', header = None)
rul_2_gt.rename(columns={0: "RUL_2gt"}, inplace = True)

column_name_dict = { 0: "engine_id", 1: "cycle", 2: "op_set_1", 3: "op_set_2", 4: "op_set_3", 5:"sensor_1", 6: "sensor_2",
                   7: "sensor_3", 8: "sensor_4", 9: "sensor_5", 10: "sensor_6", 11: "sensor_7", 12: "sensor_8", 13: "sensor_9",
                  14: "sensor_10", 15: "sensor_11", 16: "sensor_12", 17: "sensor_13", 18: "sensor_14", 19: "sensor_15", 20: "sensor_16",
                  21: "sensor_17", 22: "sensor_18", 23: "sensor_19", 24: "sensor_20", 25: "sensor_21", 26: "sensor_22", 27: "sensor_23"}

test_2 = pd.read_csv('./CMAPSSData/test_FD002.txt', header = None, sep = ' ')
test_2.rename(columns = column_name_dict, inplace = True)

train_2 = pd.read_csv('./CMAPSSData/train_FD002.txt', header = None, sep = ' ')
train_2.rename(columns = column_name_dict, inplace = True)

In [ ]:
train_2.drop(columns = ["sensor_22", "sensor_23"], inplace = True)
test_2.drop(columns = ["sensor_22", "sensor_23"], inplace = True)

In [ ]:
rul_2 = pd.DataFrame(train_1.groupby('engine_id')['cycle'].max()).reset_index()
rul_2.columns = ['engine_id', 'max']

rul_train_2 = train_2.merge(rul_2, on = ['engine_id'], how = 'left')
rul_train_2['RUL'] = rul_train_2['max'] - rul_train_2['cycle']
rul_train_2.drop(['max'], axis = 1, inplace = True)

In [ ]:
rul_train_2.head(3)

In [ ]:
rul_2_gt["engine_id"] = rul_2_gt.index + 1

max_2 = pd.DataFrame(test_2.groupby('engine_id')['cycle'].max()).reset_index()
max_2.columns = ['engine_id', 'max']
max_test_2 = test_2.merge(max_2, on=['engine_id'], how='left')
rul_test_2 = max_test_2.merge(rul_2_gt, on = ['engine_id'], how='left')

rul_test_2['RUL'] = rul_test_2['max'] - rul_test_2['cycle'] + rul_test_2["RUL_2gt"] 
rul_test_2.drop(['max', 'RUL_2gt'], axis = 1, inplace = True)

rul_train_2.groupby('engine_id')['cycle'].describe()

In [ ]:
rul_train_2.describe().transpose()

In [ ]:
df_corr = rul_train_2.drop(columns=["engine_id"]).corr()

# correlation matrix
sns.set(font_scale = 0.8)
plt.figure(figsize = (24,16))
sns.heatmap(df_corr, annot = True, fmt = ".4f",vmin = -1, vmax = 1, linewidths = .5, cmap = sns.color_palette("coolwarm", 200))

plt.figtext(.45, 0.9,'correlation matrix of train_2', fontsize = 16, ha = 'center')
plt.xticks(rotation=90)
plt.show()

In [ ]:
print("Min RUL:", rul_train_1["RUL"].min())
print("Max RUL:", rul_train_1["RUL"].max())
print(rul_train_1["RUL"].describe())

FD003 dataset

In [ ]:
rul_3_gt = pd.read_csv('./CMAPSSData/RUL_FD003.txt', header = None)
rul_3_gt.rename(columns={0: "RUL_3gt"}, inplace = True)

column_name_dict = { 0: "engine_id", 1: "cycle", 2: "op_set_1", 3: "op_set_2", 4: "op_set_3", 5:"sensor_1", 6: "sensor_2",
                   7: "sensor_3", 8: "sensor_4", 9: "sensor_5", 10: "sensor_6", 11: "sensor_7", 12: "sensor_8", 13: "sensor_9",
                  14: "sensor_10", 15: "sensor_11", 16: "sensor_12", 17: "sensor_13", 18: "sensor_14", 19: "sensor_15", 20: "sensor_16",
                  21: "sensor_17", 22: "sensor_18", 23: "sensor_19", 24: "sensor_20", 25: "sensor_21", 26: "sensor_22", 27: "sensor_23"}

test_3 = pd.read_csv('./CMAPSSData/test_FD003.txt', header = None, sep = ' ')
test_3.rename(columns = column_name_dict, inplace = True)

train_3 = pd.read_csv('./CMAPSSData/train_FD003.txt', header = None, sep = ' ')
train_3.rename(columns = column_name_dict, inplace = True)

In [ ]:
train_3.head(3)

In [ ]:
train_3.drop(columns = ["sensor_22", "sensor_23"], inplace = True)
test_3.drop(columns = ["sensor_22", "sensor_23"], inplace = True)

In [ ]:
rul_3 = pd.DataFrame(train_3.groupby('engine_id')['cycle'].max()).reset_index()
rul_3.columns = ['engine_id', 'max']

rul_train_3 = train_3.merge(rul_3, on = ['engine_id'], how = 'left')
rul_train_3['RUL'] = rul_train_3['max'] - rul_train_3['cycle']
rul_train_3.drop(['max'], axis = 1, inplace = True)

rul_train_3.head(3)

In [ ]:
rul_3_gt["engine_id"] = rul_3_gt.index + 1

max_3 = pd.DataFrame(test_3.groupby('engine_id')['cycle'].max()).reset_index()
max_3.columns = ['engine_id', 'max']
max_test_3 = test_3.merge(max_3, on = ['engine_id'], how = 'left')
rul_test_3 = max_test_3.merge(rul_3_gt, on = ['engine_id'], how = 'left')

rul_test_3['RUL'] = rul_test_3['max'] - rul_test_3['cycle'] + rul_test_3["RUL_3gt"] 
rul_test_3.drop(['max', 'RUL_3gt'], axis = 1, inplace = True)

rul_train_3.describe().transpose()

In [ ]:
deleted_3 = ["op_set_3", "sensor_1","sensor_18", "sensor_19"]

rul_train_3.drop(columns = deleted_3, inplace = True)
rul_test_3.drop(columns = deleted_3, inplace = True)

In [ ]:
df_corr = rul_train_3.drop(columns=["engine_id"]).corr()

# correlation matrix
sns.set(font_scale = 0.8)
plt.figure(figsize = (24,16))
sns.heatmap(df_corr, annot = True, fmt = ".4f",vmin = -1, vmax = 1, linewidths = .5, cmap = sns.color_palette("coolwarm", 200))

plt.figtext(.45, 0.9,'correlation matrix of train_3', fontsize = 16, ha = 'center')
plt.xticks(rotation = 90)
plt.show()

In [ ]:
print("Min RUL:", rul_train_3["RUL"].min())
print("Max RUL:", rul_train_3["RUL"].max())
print(rul_train_3["RUL"].describe())

In [ ]:
plt.plot(rul_test_3["RUL"])

In [ ]:
plt.plot(rul_test_3["RUL"])

In [ ]:
def rul_to_class_fd003(rul):
    if rul > 190:
        return 2  
    elif rul > 61:
        return 1  
    else:
        return 0 

rul_train_3["label"] = rul_train_3["RUL"].apply(rul_to_class_fd003)
rul_test_3["label"] = rul_test_3["RUL"].apply(rul_to_class_fd003)

rul_test_3.drop(columns = ["RUL"], axis = 1, inplace = True)
rul_train_3.drop(columns = ["RUL"], axis = 1, inplace = True)


rul_train_3.head(3)

In [ ]:
labels, counts = np.unique(rul_train_3["label"], return_counts = True)
ordered_labels = list(range(3))
ordered_counts = [
counts[labels.tolist().index(lbl)] if lbl in labels else 0
for lbl in ordered_labels
]

class_names = {
0: "Faulty",
1: "Transition",
2: "Healthy",
}

wrapped_labels = [textwrap.fill(class_names[i], width = 12) for i in ordered_labels]

plt.figure(figsize = (10, 6))
plt.bar(wrapped_labels, ordered_counts, color = "steelblue")
plt.title("Distribution of Combined 3-Class Operating Regimes")
plt.xlabel("Operating Regime")
plt.ylabel("Count")
plt.xticks(rotation = 30, ha = "right")  
plt.grid(axis = 'y', linestyle='--', alpha = 0.5)
plt.tight_layout()
plt.savefig("./figures/CMAPSS_03_train_labels.png")

In [ ]:
labels, counts = np.unique(rul_test_3["label"], return_counts = True)
ordered_labels = list(range(3))
ordered_counts = [
counts[labels.tolist().index(lbl)] if lbl in labels else 0
for lbl in ordered_labels
]

class_names = {
0: "Faulty",
1: "Transition",
2: "Healthy",
}

wrapped_labels = [textwrap.fill(class_names[i], width = 12) for i in ordered_labels]

plt.figure(figsize = (10, 6))
plt.bar(wrapped_labels, ordered_counts, color = "steelblue")
plt.title("Distribution of Combined 3-Class Operating Regimes")
plt.xlabel("Operating Regime")
plt.ylabel("Count")
plt.xticks(rotation = 30, ha = "right")  
plt.grid(axis = 'y', linestyle='--', alpha = 0.5)
plt.tight_layout()
plt.savefig("./figures/CMAPSS_03_test_labels.png")

In [ ]:
np.save("./CMAPSSData/train_03.npy", rul_train_3)
np.save("./CMAPSSData/test_03.npy", rul_test_3)

FD004 dataset

In [ ]:
rul_4_gt = pd.read_csv('./CMAPSSData/RUL_FD004.txt', header = None)
rul_4_gt.rename(columns = {0: "RUL_4gt"}, inplace = True)

column_name_dict = { 0: "engine_id", 1: "cycle", 2: "op_set_1", 3: "op_set_2", 4: "op_set_3", 5:"sensor_1", 6: "sensor_2",
                   7: "sensor_3", 8: "sensor_4", 9: "sensor_5", 10: "sensor_6", 11: "sensor_7", 12: "sensor_8", 13: "sensor_9",
                  14: "sensor_10", 15: "sensor_11", 16: "sensor_12", 17: "sensor_13", 18: "sensor_14", 19: "sensor_15", 20: "sensor_16",
                  21: "sensor_17", 22: "sensor_18", 23: "sensor_19", 24: "sensor_20", 25: "sensor_21", 26: "sensor_22", 27: "sensor_23"}

test_4 = pd.read_csv('./CMAPSSData/test_FD004.txt', header = None, sep = ' ')
test_4.rename(columns = column_name_dict, inplace = True)

train_4 = pd.read_csv('./CMAPSSData/train_FD004.txt', header = None, sep = ' ')
train_4.rename(columns = column_name_dict, inplace = True)

In [ ]:
train_4.head(3)

In [ ]:
train_4.drop(columns = ["sensor_22", "sensor_23"], inplace = True)
test_4.drop(columns = ["sensor_22", "sensor_23"], inplace = True)


rul_4 = pd.DataFrame(train_4.groupby('engine_id')['cycle'].max()).reset_index()
rul_4.columns = ['engine_id', 'max']

rul_train_4 = train_4.merge(rul_4, on = ['engine_id'], how = 'left')
rul_train_4['RUL'] = rul_train_4['max'] - rul_train_4['cycle']
rul_train_4.drop(['max'], axis = 1, inplace = True)

rul_train_4.head(3)

In [ ]:
rul_4_gt["engine_id"] = rul_4_gt.index + 1

max_4 = pd.DataFrame(test_4.groupby('engine_id')['cycle'].max()).reset_index()
max_4.columns = ['engine_id', 'max']
max_test_4 = test_4.merge(max_4, on = ['engine_id'], how = 'left')
rul_test_4 = max_test_4.merge(rul_4_gt, on = ['engine_id'], how = 'left')

rul_test_4['RUL'] = rul_test_4['max'] - rul_test_4['cycle'] + rul_test_4["RUL_4gt"] 
rul_test_4.drop(['max', 'RUL_4gt'], axis = 1, inplace = True)

rul_train_4.describe().transpose()

In [ ]:
df_corr = rul_train_4.drop(columns=["engine_id"]).corr()

# correlation matrix
sns.set(font_scale = 0.8)
plt.figure(figsize = (24,16))
sns.heatmap(df_corr, annot = True, fmt = ".4f",vmin = -1, vmax = 1, linewidths = .5, cmap = sns.color_palette("coolwarm", 200))

plt.figtext(.45, 0.9,'correlation matrix of train_3', fontsize = 16, ha = 'center')
plt.xticks(rotation = 90)
plt.show()

In [ ]:
print("Min RUL:", rul_train_4["RUL"].min())
print("Max RUL:", rul_train_4["RUL"].max())
print(rul_train_4["RUL"].describe())

In [ ]:
DATA_DIR = Path("./CMAPSSData")

CMAPSS_COLUMNS = (
    ["unit", "cycle", "op_set_1", "op_set_2", "op_set_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)

def make_cmapss_npy(fd_id, data_dir=DATA_DIR):

    fd = f"{fd_id:03d}"
    short = f"{fd_id:02d}"

    train_path = data_dir / f"train_FD{fd}.txt"
    test_path = data_dir / f"test_FD{fd}.txt"
    rul_path = data_dir / f"RUL_FD{fd}.txt"

    train = pd.read_csv(
        train_path,
        sep = r"\s+",
        header = None,
        names = CMAPSS_COLUMNS
    )

    test = pd.read_csv(
        test_path,
        sep = r"\s+",
        header = None,
        names = CMAPSS_COLUMNS
    )

    final_rul = pd.read_csv(
        rul_path,
        sep=r"\s+",
        header=None,
        names=["final_RUL"]
    )

    final_rul["unit"] = np.arange(1, len(final_rul) + 1)

    train = train.sort_values(["unit", "cycle"]).reset_index(drop=True)
    train["max_cycle"] = train.groupby("unit")["cycle"].transform("max")
    train["RUL"] = train["max_cycle"] - train["cycle"]
    train = train.drop(columns = ["max_cycle"])

    test = test.sort_values(["unit", "cycle"]).reset_index(drop = True)
    test["max_cycle"] = test.groupby("unit")["cycle"].transform("max")
    test = test.merge(final_rul, on = "unit", how = "left")
    test["RUL"] = test["max_cycle"] - test["cycle"] + test["final_RUL"]
    test = test.drop(columns = ["max_cycle", "final_RUL"])

    expected_cols = CMAPSS_COLUMNS + ["RUL"]
    train = train[expected_cols]
    test = test[expected_cols]

    train_arr = train.to_numpy(dtype = np.float32)
    test_arr = test.to_numpy(dtype = np.float32)

    train_out = data_dir / f"train_{short}.npy"
    test_out = data_dir / f"test_{short}.npy"

    np.save(train_out, train_arr)
    np.save(test_out, test_arr)

    print(f"FD{fd}")
    print(f"  saved: {train_out} | shape: {train_arr.shape}")
    print(f"  saved: {test_out}  | shape: {test_arr.shape}")
    print(f"  train units: {train['unit'].nunique()} | test units: {test['unit'].nunique()}")
    print(f"  train RUL range: {train['RUL'].min()} to {train['RUL'].max()}")
    print(f"  test RUL range:  {test['RUL'].min()} to {test['RUL'].max()}")
    print()

for fd_id in [2, 3, 4]:
    make_cmapss_npy(fd_id)

In [ ]:
for short in ["02", "03", "04"]:
    for split in ["train", "test"]:
        path = f"./CMAPSSData/{split}_{short}.npy"
        arr = np.load(path)

        print(path)
        print("shape:", arr.shape)
        print("dtype:", arr.dtype)
        print("first row:", arr[0])
        print("last column RUL min/max:", arr[:, -1].min(), arr[:, -1].max())
        print("NaNs:", np.isnan(arr).sum())
        print()

In [ ]:
def plot_rul_vs_time(df):

    plt.figure(figsize = (12, 6))
    for eid, group in df.groupby("engine_id"):
        plt.plot(group["cycle"], group["RUL"], alpha = 0.6, linewidth = 1)

    plt.title("RUL vs Time for Each Engine")
    plt.xlabel("Cycle")
    plt.ylabel("Remaining Useful Life (RUL)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_rul_vs_time(rul_train_3)

In [ ]:
def plot_sensor_drifts(df, sensors = None):

    if sensors is None:
        sensors = [c for c in df.columns if c.startswith("s")]

    n = len(sensors)
    fig, axes = plt.subplots(n, 1, figsize=(12, 3*n), sharex=True)

    for ax, s in zip(axes, sensors):
        for eid, group in df.groupby("engine_id"):
            ax.plot(group["cycle"], group[s], alpha=0.3)
        ax.set_title(f"Sensor Drift: {s}")
        ax.grid(True, alpha=0.3)

    plt.xlabel("Cycle")
    plt.tight_layout()
    plt.show()


In [ ]:
plot_sensor_drifts(rul_train_1)

In [ ]:
def plot_fd_comparison(fd_dict):

    stats = []
    for name, df in fd_dict.items():
        stats.append({
            "Dataset": name,
            "Num Engines": df["engine_id"].nunique(),
            "Num Cycles": len(df),
            "Num Sensors": len([c for c in df.columns if c.startswith("s")])
        })

    stats = pd.DataFrame(stats)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    sns.barplot(data = stats, x = "Dataset", y = "Num Engines", ax=axes[0])
    axes[0].set_title("Number of Engines")

    sns.barplot(data=stats, x="Dataset", y="Num Cycles", ax = axes[1])
    axes[1].set_title("Total Cycles")

    sns.barplot(data=stats, x="Dataset", y="Num Sensors", ax=axes[2])
    axes[2].set_title("Number of Sensors")

    plt.tight_layout()
    plt.show()


In [ ]:
dict = {"FD001": rul_train_1, "FD002": rul_train_2, "FD003": rul_train_3, "FD004": rul_train_4}
plot_fd_comparison(dict)

## Dataset Visualizations for the Paper

Figures for the paper's *Datasets* section, covering both the CBM/Naval propulsion dataset and all
four C-MAPSS subsets (FD001&ndash;FD004). These cells reuse the exact loading/labeling functions from
`data_pipeline.py` (not the exploratory code above, which is legacy and no longer used by the
training pipeline) so the figures match what the model actually trains on. Figures are saved to
`./figures/datasets/`.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from data_pipeline import (
    load_cbm_dataset, cbm_physical_binary_labels,
    load_cmapss_dataset, CMAPSS_SUBSETS,
)
import utils  # applies the paper-style rcParams (STIX font, palette) on import

FIG_DIR = "./figures/datasets"
os.makedirs(FIG_DIR, exist_ok=True)
PALETTE = utils.PALETTE

### CBM / Naval Propulsion Dataset

A steady-state design-of-experiments table: 16 GT sensor features per row, labeled healthy only
when both the compressor and turbine decay coefficients are at or above 0.975.

In [ ]:
CBM_PATH = "./UCI CBM Dataset/data.txt"
CBM_FEATURE_NAMES = [
    "Lever position", "Ship speed (knots)", "GT shaft torque", "GT rate of revolutions",
    "Gas generator rate of revolutions", "Starboard propeller torque", "Port propeller torque",
    "HP turbine exit temperature", "GT compressor inlet temp.", "GT compressor outlet temp.",
    "HP turbine exit pressure", "GT compressor inlet pressure", "GT compressor outlet pressure",
    "GT exhaust gas pressure", "Turbine injection control", "Fuel flow",
]

X_cbm, y_cbm_coeff = load_cbm_dataset(CBM_PATH)
y_cbm = cbm_physical_binary_labels(y_cbm_coeff, healthy_threshold=0.975)

print(f"CBM: {X_cbm.shape[0]} rows, {X_cbm.shape[1]} sensor features")
print(f"Healthy: {(y_cbm == 0).sum()}  |  Anomaly: {(y_cbm == 1).sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
healthy = y_cbm == 0
ax.scatter(y_cbm_coeff[healthy, 0], y_cbm_coeff[healthy, 1], s=6, alpha=0.5,
           color=PALETTE[0], label="Healthy")
ax.scatter(y_cbm_coeff[~healthy, 0], y_cbm_coeff[~healthy, 1], s=6, alpha=0.5,
           color=PALETTE[1], label="Anomaly")
ax.axvline(0.975, color="black", ls="--", lw=1)
ax.axhline(0.975, color="black", ls="--", lw=1)
ax.set_xlabel("Compressor decay coefficient")
ax.set_ylabel("Turbine decay coefficient")
ax.set_title("CBM: Health Label from Decay Coefficients")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "cbm_decay_scatter.png"))
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.4))
counts = [(y_cbm == 0).sum(), (y_cbm == 1).sum()]
ax.bar(["Healthy", "Anomaly"], counts, color=[PALETTE[0], PALETTE[1]], edgecolor="white")
for i, c in enumerate(counts):
    ax.text(i, c, f"{c}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Rows")
ax.set_title("CBM: Class Balance")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "cbm_class_balance.png"))
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(12, 10))
for i, ax in enumerate(axes.ravel()):
    ax.hist(X_cbm[healthy, i], bins=30, alpha=0.55, color=PALETTE[0], label="Healthy")
    ax.hist(X_cbm[~healthy, i], bins=30, alpha=0.55, color=PALETTE[1], label="Anomaly")
    ax.set_title(CBM_FEATURE_NAMES[i], fontsize=8)
    ax.tick_params(labelsize=6)
axes[0, 0].legend(fontsize=7)
fig.suptitle("CBM: Sensor Feature Distributions by Health Label")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "cbm_sensor_distributions.png"))
plt.show()

In [ ]:
with np.errstate(invalid="ignore", divide="ignore"):
    corr = np.corrcoef(X_cbm, rowvar=False)  # NaN rows/cols mean a constant (zero-variance) sensor

fig, ax = plt.subplots(figsize=(7, 6.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(CBM_FEATURE_NAMES)))
ax.set_yticks(range(len(CBM_FEATURE_NAMES)))
ax.set_xticklabels(CBM_FEATURE_NAMES, rotation=90, fontsize=6)
ax.set_yticklabels(CBM_FEATURE_NAMES, fontsize=6)
ax.set_title("CBM: Sensor Correlation Matrix")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "cbm_correlation_heatmap.png"))
plt.show()

### C-MAPSS Turbofan Degradation Dataset (FD001&ndash;FD004)

Four subsets of run-to-failure aircraft-engine trajectories, differing in operating-condition count
(ONE for FD001/FD003, SIX for FD002/FD004) and fault-mode count (ONE vs. TWO). RUL is computed
directly from the raw text files by `data_pipeline.load_cmapss_dataset` &mdash; no intermediate
`.npy` file involved.

In [ ]:
cmapss_data = {}
print(f"{'Subset':<8}{'Train engines':>15}{'Test engines':>15}{'Traj. length (min-mean-max)':>32}")
for fd in CMAPSS_SUBSETS:
    X_train, y_train, unit_train, X_test, y_test, unit_test = load_cmapss_dataset(fd=fd)
    cmapss_data[fd] = dict(
        X_train=X_train, y_train=y_train, unit_train=unit_train,
        X_test=X_test, y_test=y_test, unit_test=unit_test,
    )
    lengths = np.array([np.sum(unit_train == u) for u in np.unique(unit_train)])
    print(f"{fd:<8}{len(np.unique(unit_train)):>15}{len(np.unique(unit_test)):>15}"
          f"{f'{lengths.min()}-{lengths.mean():.0f}-{lengths.max()}':>32}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8.5, 8))
for ax, fd in zip(axes.ravel(), CMAPSS_SUBSETS):
    X = cmapss_data[fd]["X_train"]
    ax.scatter(X[:, 0], X[:, 1], s=3, alpha=0.25, color=PALETTE[2])
    ax.set_title(fd)
    ax.set_xlabel("Operational setting 1")
    ax.set_ylabel("Operational setting 2")
fig.suptitle("C-MAPSS: Operating Conditions per Subset\n(FD001/FD003 = one condition, FD002/FD004 = six)")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "cmapss_operating_conditions.png"))
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8.5, 7))
for ax, fd in zip(axes.ravel(), CMAPSS_SUBSETS):
    d = cmapss_data[fd]
    ax.hist(d["y_train"], bins=40, alpha=0.55, color=PALETTE[0], label="Train")
    ax.hist(d["y_test"], bins=40, alpha=0.55, color=PALETTE[3], label="Test")
    ax.axvline(100, color=PALETTE[2], ls="--", lw=1, label="Normal cutoff (100)")
    ax.axvline(30, color=PALETTE[1], ls="--", lw=1, label="Anomaly cutoff (30)")
    ax.set_title(fd)
    ax.set_xlabel("RUL (cycles)")
    ax.set_ylabel("Count")
axes[0, 0].legend(fontsize=7)
fig.suptitle("C-MAPSS: RUL Distribution per Subset")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "cmapss_rul_distributions.png"))
plt.show()

In [ ]:
def top_drifting_sensors(X, y, k=3):
    """Sensor columns (skipping the 3 operational settings) ranked by |correlation| with RUL."""
    sensor_cols = list(range(3, X.shape[1]))
    with np.errstate(invalid="ignore"):
        corrs = [np.corrcoef(X[:, c], y)[0, 1] for c in sensor_cols]
    corrs = [0.0 if np.isnan(c) else c for c in corrs]
    order = np.argsort(-np.abs(corrs))[:k]
    return [sensor_cols[i] for i in order], [f"sensor_{sensor_cols[i] - 2}" for i in order]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# FD001: one operating condition, so the top RUL-correlated sensors trend cleanly to failure.
X1, unit1, y1 = cmapss_data["FD001"]["X_train"], cmapss_data["FD001"]["unit_train"], cmapss_data["FD001"]["y_train"]
cols, names = top_drifting_sensors(X1, y1)
u1 = unit1[0]
Xu1 = X1[unit1 == u1]
cycles1 = np.arange(1, len(Xu1) + 1)
for col, name, color in zip(cols, names, PALETTE):
    z = (Xu1[:, col] - Xu1[:, col].mean()) / (Xu1[:, col].std() + 1e-8)
    axes[0].plot(cycles1, z, lw=1.1, color=color, label=name)
axes[0].set_title(f"FD001, engine {u1} -- top RUL-correlated sensors")
axes[0].set_xlabel("Cycle")
axes[0].set_ylabel("Z-scored sensor reading")
axes[0].legend(fontsize=7)

# FD004: six operating conditions. Its best raw-correlated sensor still barely correlates with
# RUL (|r|~0.08 vs ~0.7 for FD001) because condition switching, not degradation, dominates the
# raw signal -- overlaying the raw operating setting makes that directly visible, and is itself
# the core reason this project treats regime and degradation separately rather than pooling them.
X4, unit4, y4 = cmapss_data["FD004"]["X_train"], cmapss_data["FD004"]["unit_train"], cmapss_data["FD004"]["y_train"]
col4, name4 = top_drifting_sensors(X4, y4, k=1)
u4 = unit4[0]
Xu4 = X4[unit4 == u4]
cycles4 = np.arange(1, len(Xu4) + 1)
z4 = (Xu4[:, col4[0]] - Xu4[:, col4[0]].mean()) / (Xu4[:, col4[0]].std() + 1e-8)

ax2 = axes[1].twinx()
ax2.plot(cycles4, Xu4[:, 0], color=PALETTE[3], alpha=0.6, lw=0.8)
ax2.set_ylabel("Operational setting 1 (raw)", color=PALETTE[3], fontsize=8)
ax2.tick_params(axis="y", labelcolor=PALETTE[3])
axes[1].plot(cycles4, z4, lw=1.0, color="black", zorder=3)
axes[1].set_title(f"FD004, engine {u4} -- {name4[0]} tracks the operating condition")
axes[1].set_xlabel("Cycle")
axes[1].set_ylabel("Z-scored sensor reading")

fig.suptitle("C-MAPSS: Example Sensor Trajectories to Failure")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "cmapss_sensor_trajectories.png"))
plt.show()

In [ ]:
fd_ref = "FD001"
X_ref = cmapss_data[fd_ref]["X_train"]
sensor_labels = ["op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]

with np.errstate(invalid="ignore", divide="ignore"):
    corr = np.corrcoef(X_ref, rowvar=False)  # NaN rows/cols mean a constant (zero-variance) sensor

fig, ax = plt.subplots(figsize=(8, 7.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(sensor_labels)))
ax.set_yticks(range(len(sensor_labels)))
ax.set_xticklabels(sensor_labels, rotation=90, fontsize=6)
ax.set_yticklabels(sensor_labels, fontsize=6)
ax.set_title(f"C-MAPSS {fd_ref}: Feature Correlation Matrix")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "cmapss_fd001_correlation_heatmap.png"))
plt.show()